## 罰則項なし線形回帰のクロスバリデーションによる線形回帰

簡単な例で罰則項なし線形回帰とクロスバリデーションによる性能評価を行います。

最初の部分は同じなので説明を省きます。

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

# pandas表示設定
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 80)

In [ ]:
g_data_name = "x5_sin"  # x5_sin, x123
g_normalizationtype = "standard"
g_regtype = "linear" # linear, lasso, ridge
g_shuffle = True # shuffle or not in CV

In [ ]:
def get_data(data_name):
    """観測データの作成

    Args:
        data_name (str): 作成するデータの名前。

    Raises:
        ValueError: 規定外のdata_name。

    Returns:
        pd.DataFrame: 観測データ。
        pd.DataFrame: 新規データ
        List(str): 説明変数名のリスト
        str: 目的変数
    """    
    if data_name == "x5_sin":
        filename = "../data_calculated/x5_sin.csv"
        filename_new = "../data_calculated/x5_sin_new.csv"
        descriptor_names = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6']
        # descriptor_names = ['x1', 'x2', 'x3', 'x4', 'x5', ]
        target_name = 'y'
    elif data_name == "x123":
        filename = "../data_calculated/x123.csv"
        filename_new = "../data_calculated/x123_new.csv"
        descriptor_names = ['x1', 'x2', 'x3']
        target_name = 'y'
    else:
        raise ValueError("unknown data_name={}".format(data_name))
    df_obs = pd.read_csv(filename)
    df_new = pd.read_csv(filename_new)
    return df_obs, df_new, descriptor_names, target_name

g_df_obs, g_df_new, g_descriptor_names, g_target_name = get_data(g_data_name)

g_df_obs

In [ ]:
# obs
g_Xraw = g_df_obs.loc[:, g_descriptor_names].values
g_y = g_df_obs.loc[:, g_target_name].values

# new 
g_Xraw_new = g_df_new.loc[:, g_descriptor_names].values
g_y_new = g_df_new.loc[:, g_target_name].values

In [ ]:
def scale_X(Xraw, normalizationtype=None, scaler=None):
    """Xを規格化する。

    Args:
        Xraw (np.ndarray): 説明変数。
        normalizationtype (str, optional): 規格化の名前. Defaults to None.
        scaler (StandardScaler|MinMaxScaler, optional): 規格化クラスインスタンス. Defaults to None.

    Raises:
        ValueError: 規定外normalizationtype

    Returns:
        nd.ndarray: 規格化された説明変数

    """    
    if scaler is not None:
        print("use", scaler)
        X = scaler.transform(Xraw)
    else:
        print("normalizationtype", normalizationtype)
        if normalizationtype=="standard":
            from sklearn.preprocessing import StandardScaler
            scaler = StandardScaler()
            scaler.fit(Xraw)
            X = scaler.transform(Xraw)    
        elif normalizationtype=="minmax":
            from sklearn.preprocessing import MinMaxScaler
            scaler = MinMaxScaler()
            scaler.fit(Xraw)
            X = scaler.transform(Xraw)    
        elif normalizationtype is None:
            # 規格化を行わない。
            X = Xraw
            scaler = None
        else:
            raise ValueError("unkown normalizationtype={}".format(normalizationtype))
    return X, scaler


g_X, g_scaler = scale_X(g_Xraw, g_normalizationtype)
g_X_new, _ = scale_X(g_Xraw_new, scaler=g_scaler)

In [ ]:
plt.plot(g_X)
plt.show()
plt.plot(g_X_new)
_ # <- plot.showの戻り値の表示をしないために追加している。

choose_linear_model()で用いる線型回帰モデルの定義を行います。

KFoldを用いてクロスバリデーション（CV)の処理を行います。

ここでは最後にテストデータに対するCVスコアの平均と標準偏差を出力しています。

In [ ]:
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.model_selection import KFold

def choose_linear_model(regtype :str, alpha:float=1e-2):
    """線形モデルの選択を行う

    Args:
        regtype (str): 線形モデル名
        alpha (float, optional): Lasso, Ridgeのhyperparameter. Defaults to 1e-2.

    Raises:
        ValueError: 規定外線形モデル名。

    Returns:
        LinearRegression|Lasso|Ridge: 線型回帰モデルinstance
    """
    if regtype=="linear":
        reg = LinearRegression()
    elif regtype=="lasso":
        reg = Lasso(alpha=alpha)
    elif regtype=="ridge":
        reg = Ridge(alpha=alpha)
    else:
        raise ValueError("unkown regtype={}".format(regtype))
    return reg

def linear_regression_CV_score(X, y, regtype="linear", 
                               n_splits=10, shuffle=True, random_state=1):
    """linear regression with cross validation sore

    Args:
        X (np.array): descriptor
        y (np.array): target variable
        n_splits (int, optional): the number of splits in CV. Defaults to 10.
        random_state (int, optional): random state in KFold(). Defaults to 1.

    Returns:
        dict: the mean value of the CV score, the stddev value of the CV score
        LinearRegression|Lasso|Ridge: 線型回帰モデルinstance
    """
    reg = choose_linear_model(regtype)
    kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)

    test_score_list = []
    for train, test in kf.split(X):
        Xtrain, ytrain = X[train], y[train]
        Xtest, ytest = X[test], y[test]
        reg.fit(Xtrain, ytrain)

        test_score = reg.score(Xtest, ytest)
        test_score_list.append(test_score)

    return {"mean(R2)":np.mean(test_score_list), "std(R2)":np.std(test_score_list)}, reg

g_result, g_reg = linear_regression_CV_score(g_X, g_y, g_regtype, shuffle=g_shuffle)
g_result

ytestpも同時に出力するには以下のように書きます。

In [ ]:
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold


def linear_regression_CV_score_ytestp(X, y, regtype="linear", 
                                      n_splits=10, random_state=1):
    """linear regression with cross validation sore.
        It also returns y_test and y_test^predict

    Args:
        X (np.array): explanatory variables
        y (np.array): target variable
        n_splits (int, optional): the number of splits in CV. Defaults to 10.
        random_state (int, optional): random state in KFold(). Defaults to 1.

    Returns:
        dict: the mean value of the CV score, the stddev value of the CV score, 
            a list of y_test,a list of y_test^predict.
        LinearRegression|Lasso|Ridge: 線型回帰モデルinstance
    """
    reg = choose_linear_model(regtype)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    test_score_list = []
    ytest_list = []
    ytestp_list = []
    for train, test in kf.split(X):
        Xtrain, ytrain = X[train], y[train]
        Xtest, ytest = X[test], y[test]
        reg.fit(Xtrain, ytrain)

        ytestp = reg.predict(Xtest)
        ytest_list.append(ytest)
        ytestp_list.append(ytestp)

        test_score = r2_score(ytest, ytestp)
        test_score_list.append(test_score)

    return {"mean(R2)":np.mean(test_score_list), "std(R2)":np.std(test_score_list), \
           "ytest": ytest_list, "ytestp": ytestp_list}, reg

g_result, g_reg = linear_regression_CV_score_ytestp( g_X, g_y, regtype=g_regtype)

for _key in ["mean(R2)","std(R2)"]:
    print(_key,":",g_result[_key])


### 新規データに対する予測

In [ ]:
g_yp_new = g_reg.predict(g_X_new)

## 可視化

CVのテストデータのindexは以下のように指定されています。

まず、shuffle=Falseの場合です。
各CV分割でテストデータ以外のデータは訓練データとなります。

In [ ]:
def show_CV_splot(X, shuffle, n_splits = 10, random_state=0):
    """CVの分離具合を表示する。

    Args:
        X (np.ndarray): 説明変数
        shuffle (bool): KFoldでのshffle
        n_splits (int, optional): KFoldでの分割数. Defaults to 10.
        random_state (int, optional): KFoldでのrandom_state. Defaults to 0.
    """
    if shuffle:
        kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
    else:
        kf = KFold(n_splits=n_splits, shuffle=shuffle)        
        
    for i, (train, test) in enumerate(kf.split(X)):
        # test選択部分に色を付けて表示しているだけ。
        c = np.zeros(test.shape[0])
        c += i
        plt.plot(test, c, "o")
    plt.xlabel("index")
    plt.ylabel("CV id")
    plt.show()
    
show_CV_splot(g_X, shuffle=False)

shuffle = Trueと呼ぶとランダムに順序を並び替えます。

In [ ]:
show_CV_splot(g_X, shuffle=True)


($y^{obs}$, $y^{pred}$)を表示します。
異なるCV setは異なった色で表示されます。

In [ ]:
def plot_y_yp(y,yp, title: str=None):
    """y vs ypを図示する。

    Args:
        y (np.ndarray): 目的変数観測値
        yp (np.ndarray): s目的変数予測値
        title (str, optional): 図のtitle. Defaults to None.
    """
    fig, ax = plt.subplots(figsize=(5,5))

    # $y^{obs}$ vs $y^{predict}$
    ax.plot(y,yp,"o")

    # 斜め線を引く
    yall = np.hstack([y,yp])
    ylim = yall.min(), yall.max()
    ax.plot(ylim,ylim,"--")

    # labelを書く
    ax.set_xlabel("$y_{obs}$")
    ax.set_ylabel("$y_{pred}$")
    if title is not None:
        ax.set_title(title)
    fig.show()

plot_y_yp(g_result["ytest"],g_result["ytestp"],)

In [ ]:
# 新規データに対する予測
plot_y_yp(g_y_new, g_yp_new)

線形回帰モデル係数の表示を行う。

In [ ]:
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold


def linear_regression_CV_coef(X, y, regtype="linear",
                              n_splits=10, random_state=1):
    """linear regression with cross validation

    Args:
        X (np.array): explanatory variables
        y (np.array): target variable
        n_splits (int, optional): the number of splits in CV. Defaults to 10.
        random_state (int, optional): random state in KFold(). Defaults to 1.

    Returns:
        list: a list of linear coefficients
    """
    reg = choose_linear_model(regtype)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    coef_list = []
    for train, test in kf.split(X):
        Xtrain, ytrain = X[train], y[train]
        Xtest, ytest = X[test], y[test]
        reg.fit(Xtrain, ytrain)
        coef_list.append(list(reg.coef_.ravel()))
    return coef_list


g_coef_list = linear_regression_CV_coef(g_X, g_y, regtype=g_regtype)


In [ ]:
g_coef_list

係数の表示を行う。

regtype="linear"の場合は、ハイパーパラメタは回帰モデルにありませんから、訓練データ、テストデータの組み合わせを変えただけです。

In [ ]:
def show_coeflist(coeflist, data_name, regtype):
    """線形モデルのcoefの図示。

    Args:
        coeflist ([float]): 回帰係数
        data_name (str)): データ名
        regtype (LinearModel): 線形モデルインスタンス
    """
    fig, ax = plt.subplots()
    dfcoef = pd.DataFrame(coeflist)
    dfcoef.plot(ax=ax)
    ax.set_xlabel("CV set index")
    ax.set_title("{},{}".format(data_name,regtype))
    fig.tight_layout()
    import os
    os.makedirs("image_executed", exist_ok=True)
    fig.savefig("image_executed/fig_regression_CV_{}_{}.png".format(data_name,regtype))
    
show_coeflist(g_coef_list, g_data_name, g_regtype)

予め計算しておいた結果を示します。上から罰則項無し線形回帰モデル,Lasso,Ridgeで、縦軸が係数、横軸がCVのindexを示します。

![図：罰則項無し線形回帰モデルの係数の変化](image_keep/fig_regression_CV_x5_sin_linear.png)
![図：Lassoの係数の変化](image_keep/fig_regression_CV_x5_sin_lasso.png)
![図：Ridgeの係数の変化](image_keep/fig_regression_CV_x5_sin_ridge.png)


regtype="linear"では実は訓練データが異なると桁が異なる係数を持つ線形モデルができていました。
説明変数間に（近似的な）共線形性があるので、
過学習を起こし、全く異なるモデルができています。

### 問題１

regtypeをlasso, ridgeに変えて実行する。
この際にWarningが出るかも知れませんが、問題ないので無視してください。


### 問題２

Ridge回帰, LASSOで最適なhyperparameterの選択ができるようになった後に、このscriptのLinearRegression()をRidge(alpha=alpha_opt), Lasso(alpha=alpha_optに置き換えて実行する。